# 📡 Customer Churn Prediction

**Goal:** Predict which telecom customers are likely to churn using machine learning.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — 7,043 customers, 21 features

**Approach:**
1. Exploratory Data Analysis (EDA)
2. Data Preprocessing & Feature Engineering
3. Model Training & Comparison (Logistic Regression, Random Forest, XGBoost)
4. Handling Class Imbalance
5. Model Explainability with SHAP

---

## 1. Setup & Imports

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from xgboost import XGBClassifier

# Global plot style
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
# Download dataset from Kaggle
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Dataset path:", path)

In [ ]:
# Load into DataFrame and preview
df = pd.read_csv(path + "/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.sample(5)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic dataset overview: shape, dtypes, missing values, duplicates
print(df.info())
print("=" * 50)
print(df.describe())
print("=" * 50)
print("Missing values:\n", df.isnull().sum())
print("=" * 50)
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Unique value counts per column — helps identify binary vs multi-class features
df.nunique()

In [ ]:
# Unique values for all categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

In [ ]:
# Visual EDA: churn rate, numerical distributions, and key categorical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Customer Churn — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Churn distribution
churn_counts = df['Churn'].value_counts()
axes[0, 0].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
               colors=['#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Churn Rate')

# 2. Tenure distribution by churn status
df.groupby('Churn')['tenure'].plot(kind='hist', ax=axes[0, 1], alpha=0.7,
                                    bins=30, color=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Tenure by Churn')
axes[0, 1].set_xlabel('Tenure (months)')
axes[0, 1].legend(['No Churn', 'Churn'])

# 3. Monthly charges distribution by churn status
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[0, 2],
            palette=['#2ecc71', '#e74c3c'])
axes[0, 2].set_title('Monthly Charges vs Churn')

# 4. Contract type vs churn
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Contract Type vs Churn')
axes[1, 0].tick_params(rotation=15)

# 5. Internet service type vs churn
internet_churn = df.groupby(['InternetService', 'Churn']).size().unstack()
internet_churn.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Internet Service vs Churn')
axes[1, 1].tick_params(rotation=15)

# 6. Payment method vs churn
payment_churn = df.groupby(['PaymentMethod', 'Churn']).size().unstack()
payment_churn.plot(kind='bar', ax=axes[1, 2], color=['#2ecc71', '#e74c3c'])
axes[1, 2].set_title('Payment Method vs Churn')
axes[1, 2].tick_params(rotation=20)

plt.tight_layout()
plt.show()

### EDA Key Insights

| Feature | Observation |
|---|---|
| **Churn Rate** | 26.5% — dataset is imbalanced |
| **Tenure** | New customers (0–10 months) churn the most |
| **MonthlyCharges** | Churned customers pay ~$15 more on average |
| **Contract** | Month-to-month contracts have the highest churn |
| **InternetService** | Fiber optic customers churn more than DSL |
| **PaymentMethod** | Electronic check users have the highest churn rate |

## 4. Data Preprocessing

In [ ]:
# --- Step 1: Drop customerID — not a predictive feature
df = df.drop(columns=['customerID'])

# --- Step 2: Fix TotalCharges dtype
# TotalCharges is stored as string due to hidden whitespace values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Fill NaN with MonthlyCharges (these are new customers with tenure=0)
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

# --- Step 3: Encode binary columns to 0/1
binary_map = {'Yes': 1, 'No': 0, 'Female': 1, 'Male': 0}
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = df[col].map(binary_map)

# --- Step 4: Map "No service" variants to 0
# Customers without internet/phone simply don't have those services — treated as 0
no_service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in no_service_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0,
                           'No internet service': 0,
                           'No phone service': 0})

# --- Step 5: One-Hot Encode multi-class categorical columns
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'],
                    drop_first=False)

# --- Step 6: Convert bool columns to int (get_dummies creates bool by default)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"Final shape: {df.shape}")
print(df.dtypes)

In [ ]:
# Preview preprocessed data
df.head(5)

## 5. Feature Selection via Correlation Analysis

In [ ]:
# Correlation heatmap — all features vs all features
plt.figure(figsize=(14, 10))
corr = df.corr()

sns.heatmap(corr,
            annot=True,
            fmt='.2f',
            cmap='RdYlGn',
            center=0,
            linewidths=0.5,
            annot_kws={'size': 7})

plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with target variable sorted by absolute value
print("\nFeature correlation with Churn (sorted by absolute value):")
churn_corr = corr['Churn'].drop('Churn')
print(churn_corr.reindex(churn_corr.abs().sort_values(ascending=False).index))

In [ ]:
# Drop features with near-zero correlation to Churn (|corr| < 0.02)
# Note: negative correlation is still a valid signal — we use absolute value
churn_corr = df.corr()['Churn'].drop('Churn')
weak_features = churn_corr[churn_corr.abs() < 0.02].index.tolist()

print("Dropped weak features (|corr| < 0.02):", weak_features)

df = df.drop(columns=weak_features)
print(f"Remaining features: {df.shape[1] - 1}")

## 6. Train / Test Split

In [ ]:
# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# 80/20 split — stratify ensures same churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\nChurn rate in train: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test:  {y_test.mean()*100:.1f}%")

## 7. Model Training & Evaluation

We train and compare three models:
- **Logistic Regression** — simple, interpretable baseline
- **Random Forest** — ensemble tree model
- **XGBoost (Balanced)** — gradient boosting with class imbalance handling

### 7.1 Logistic Regression

In [ ]:
# Scale features — required for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}")

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('Logistic Regression — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### 7.2 Random Forest

In [ ]:
# Train Random Forest — no scaling needed for tree-based models
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}")

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### 7.3 XGBoost with Class Balancing

The dataset is imbalanced (73.5% vs 26.5%). We use `scale_pos_weight` to penalize misclassification of the minority class (churned customers).

In [ ]:
# Calculate class weight ratio to handle imbalanced data
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()
scale_pos_weight = negative / positive
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Train XGBoost with class balancing
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("XGBOOST (BALANCED) RESULTS")
print("=" * 50)
print(classification_report(y_test, y_pred_xgb, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb):.4f}")

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('XGBoost (Balanced) — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 8. Model Comparison

In [ ]:
# Build a summary comparison table for all three models
from sklearn.metrics import f1_score, precision_score, recall_score

results = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Random Forest':       (y_pred_rf, y_prob_rf),
    'XGBoost (Balanced)':  (y_pred_xgb, y_prob_xgb),
}

summary = []
for name, (y_pred, y_prob) in results.items():
    summary.append({
        'Model':          name,
        'ROC-AUC':        round(roc_auc_score(y_test, y_prob), 4),
        'Churn Recall':   round(recall_score(y_test, y_pred), 4),
        'Churn Precision':round(precision_score(y_test, y_pred), 4),
        'Churn F1':       round(f1_score(y_test, y_pred), 4),
    })

summary_df = pd.DataFrame(summary).set_index('Model')
print(summary_df.to_string())

**Selected Model: XGBoost (Balanced)**

Although Logistic Regression achieves similar ROC-AUC, XGBoost (Balanced) identifies **94 more churned customers** in the test set thanks to `scale_pos_weight`. From a business perspective, missing a churned customer is far more costly than a false positive.

## 9. Model Explainability — SHAP

In [ ]:
# SHAP explains why the model makes each prediction
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Global feature importance — which features drive churn predictions the most
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)

In [ ]:
# Detailed SHAP plot — shows direction of impact (positive = increases churn)
shap.summary_plot(shap_values, X_test, show=True)

## 10. Business Insights & Recommendations

Based on SHAP feature importance and EDA findings:

| # | Driver | Recommendation |
|---|---|---|
| 1 | **Contract type** — month-to-month has highest churn | Incentivize customers to switch to annual contracts |
| 2 | **Tenure** — new customers churn most in first 10 months | Launch an onboarding loyalty program for new users |
| 3 | **Fiber optic** — higher churn than DSL users | Investigate service quality issues for fiber customers |
| 4 | **Monthly charges** — higher bills → higher churn | Offer personalized discounts to high-value at-risk customers |
| 5 | **Electronic check** — highest churn among payment methods | Encourage auto-payment enrollment with incentives |

---

**Model performance summary:**
- ROC-AUC: **0.84** — strong discriminative power
- Churn Recall: **0.79** — catches 79% of customers who would have churned
- The model can flag at-risk customers **before** they leave, enabling proactive retention campaigns